# Infinite-Width Networks

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 11.03 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/11_Functional_Analysis/04_infinite_width_networks.ipynb)

---

## 🎯 Learning Objective

Study mean-field theory, signal propagation, and the edge-of-chaos regime that governs infinite-width network behavior.

---

## 📐 Theory

`11.03`'s NTK theory describes what happens to the network's *gradients* in the infinite-width
limit. This closing notebook studies the simpler, earlier question that theory was originally
built to answer: what happens to the *signal itself* (activations) as it propagates forward
through an increasingly deep, increasingly wide network at initialization — and why does the
answer determine whether that network can be trained at all.

### Mean-field theory: tracking one number instead of a whole activation vector

In an infinitely wide layer, the activations become (by an argument reminiscent of the Central
Limit Theorem, recall `03.06`) well-approximated by i.i.d. Gaussian random variables. This
**mean-field** simplification means you don't need to track a whole width-$n$ activation vector
through the network — just a couple of scalar statistics (its variance, and its correlation with
another input's activation) evolving layer-to-layer according to a deterministic recursion. This
notebook verifies that recursion's key qualitative predictions empirically, on genuinely wide
(but finite) simulated networks, rather than deriving the recursion's exact closed form.

### Signal propagation: variance across depth

For a layer $z = Wx + b$ with $W_{ij}\sim\mathcal{N}(0,\sigma_w^2/n)$ i.i.d. (the $1/n$ scaling
is exactly `11.03`'s NTK-parameterization idea, applied to keep pre-activation variance from
blowing up with width), the pre-activation variance evolves as $q^{(\ell+1)} = \sigma_w^2\,
\mathbb{E}[\phi(z)^2] + \sigma_b^2$ for activation $\phi$. Whether this recursion has a stable
fixed point — and whether that fixed point is reached quickly or slowly — determines whether
signal *vanishes* (variance $\to 0$, every unit's activation collapses to the same constant,
destroying all information) or *explodes* (variance $\to\infty$, numerical overflow) as depth
grows. Below, we verify He initialization's specific value $\sigma_w^2=2$ (for ReLU) is
precisely the value that keeps variance roughly constant across depth, while noticeably smaller
or larger values cause exactly the vanishing/exploding behavior just described.

### The correlation map and the order/chaos phase transition

A second, independent recursion tracks $c^{(\ell)}$, the correlation between two *different*
inputs' activations at layer $\ell$. For a saturating activation like $\tanh$, this recursion
has two qualitatively different long-run behaviors depending on $\sigma_w^2$: an **ordered
phase** where $c^{(\ell)}\to 1$ as depth grows (any two distinct inputs become
indistinguishable — the network collapses all information), and a **chaotic phase** where
$c^{(\ell)}$ converges to some value below 1 (inputs stay distinguishable, but nearby inputs'
outputs decorrelate unpredictably with depth). Between them sits the **edge of chaos**: the
critical $\sigma_w^2$ where the correlation map's fixed point at $c=1$ transitions from
attracting to repelling. Below, we map out this transition directly by propagating pairs of
correlated random vectors through networks at several $\sigma_w^2$ values and plotting how their
correlation evolves with depth — the qualitative signature of order vs. chaos is visible without
needing the recursion's exact closed form.

### Dynamical isometry (brief note)

**Dynamical isometry** is a stronger condition than merely "the edge of chaos": it asks that the
*entire spectrum* of the input-output Jacobian be close to 1 (not just its mean squared value),
which requires careful joint control of both the weight distribution and the activation function
together (e.g. orthogonal weight init) — full treatment is beyond this notebook's scope, but the
underlying motivation is identical: well-conditioned signal propagation, at every layer, not
just on average.

### Why this determines trainability

A network stuck in the ordered phase has activations (and, by a closely related argument,
gradients) that shrink geometrically with depth — the same vanishing-gradient phenomenon `06`
introduced concretely for deep networks. Below, we close the loop: train real (finite but deep)
networks at several $\sigma_w^2$ values and confirm directly that the one deep in the ordered
phase fails to learn at all (loss stuck near its initial value, near-zero gradient), while
initializations at or past the edge of chaos train successfully — turning "edge of chaos" from
an abstract phase-transition concept into a directly observable trainability threshold.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Propagating two initially weakly-correlated random vectors through a `tanh` network at several
values of $\sigma_w^2$ and tracking how their correlation evolves with depth makes the
order/chaos phase transition directly visible: small $\sigma_w^2$ drives correlation to 1
(information collapse), large $\sigma_w^2$ keeps it well below 1.

In [ ]:
rng_viz = np.random.default_rng(0)

def propagate_correlation(sigma_w_sq, sigma_b_sq, n_layers, width, activation, c0, n_trials, rng):
    correlations = []
    for trial in range(n_trials):
        x1 = rng.normal(size=width)
        x1 = x1 / np.linalg.norm(x1) * np.sqrt(width)
        noise = rng.normal(size=width)
        noise = noise / np.linalg.norm(noise) * np.sqrt(width)
        x2 = c0 * x1 + np.sqrt(max(1 - c0**2, 0)) * noise
        layer_corrs = [np.corrcoef(x1, x2)[0, 1]]
        for layer in range(n_layers):
            W = rng.normal(scale=np.sqrt(sigma_w_sq / width), size=(width, width))
            b = rng.normal(scale=np.sqrt(sigma_b_sq), size=width)
            z1, z2 = W @ x1 + b, W @ x2 + b
            x1, x2 = activation(z1), activation(z2)
            layer_corrs.append(np.corrcoef(z1, z2)[0, 1])
        correlations.append(layer_corrs)
    return np.mean(correlations, axis=0)

width, n_layers, n_trials = 250, 20, 25
sigma_w_values = [0.8, 1.05, 1.3, 3.0]
colors = ["#C44E52", "#DD8452", "#55A868", "#4C72B0"]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
final_corrs = {}
for sigma_w_sq, color in zip(sigma_w_values, colors):
    corr_traj = propagate_correlation(sigma_w_sq, 0.05, n_layers, width, np.tanh, c0=0.3, n_trials=n_trials, rng=rng_viz)
    final_corrs[sigma_w_sq] = corr_traj[-1]
    ax.plot(corr_traj, color=color, lw=2, marker='o', ms=4, label=f"$\\sigma_w^2$={sigma_w_sq}")

ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel("layer depth"); ax.set_ylabel("correlation between two inputs' activations")
ax.set_title("Order/chaos phase transition in a deep tanh network:\nsmall $\\sigma_w^2$ collapses correlation to 1, large keeps it well below")
ax.legend()
plt.show()

print("Final correlation (after", n_layers, "layers), starting from c0=0.3:")
for sigma_w_sq in sigma_w_values:
    print(f"  sigma_w^2={sigma_w_sq}: final correlation = {final_corrs[sigma_w_sq]:.4f}")
print(f"\nSmall sigma_w^2 (ordered phase) drives correlation toward 1 -- any two DIFFERENT")
print(f"inputs become indistinguishable by the last layer. Large sigma_w^2 (chaotic phase)")
print(f"keeps correlation well below 1, preserving SOME ability to distinguish inputs.")

## 🐍 Implementation from Scratch

We verify He initialization's specific value keeps ReLU pre-activation variance roughly
constant across depth (while smaller/larger values cause genuine vanishing/exploding), then
quantify the order/chaos correlation map's phase transition precisely by checking how many
layers it takes to reach near-total correlation collapse at each $\sigma_w^2$.

In [ ]:
rng = np.random.default_rng(0)

def propagate_variance(sigma_w_sq, sigma_b_sq, n_layers, width, activation, x0_var, n_trials, rng):
    # Tracks a SINGLE unit's pre-activation variance across layers, for a wide network at
    # initialization -- the empirical quantity mean-field theory predicts a recursion for.
    variances = []
    for trial in range(n_trials):
        x = rng.normal(scale=np.sqrt(x0_var), size=width)
        layer_vars = [np.var(x)]
        for layer in range(n_layers):
            W = rng.normal(scale=np.sqrt(sigma_w_sq / width), size=(width, width))
            b = rng.normal(scale=np.sqrt(sigma_b_sq), size=width)
            z = W @ x + b
            x = activation(z)
            layer_vars.append(np.var(z))
        variances.append(layer_vars)
    return np.mean(variances, axis=0)

relu = lambda z: np.maximum(z, 0)
width, n_layers, n_trials = 300, 25, 30

var_he = propagate_variance(2.0, 0.0, n_layers, width, relu, 1.0, n_trials, rng)   # He's own sigma_w^2=2 for ReLU
var_low = propagate_variance(0.5, 0.0, n_layers, width, relu, 1.0, n_trials, rng)
var_high = propagate_variance(8.0, 0.0, n_layers, width, relu, 1.0, n_trials, rng)

print("Pre-activation variance across depth (every 5 layers):")
print(f"  He init (sigma_w^2=2.0, the textbook ReLU value): {np.round(var_he[::5], 4)}")
print(f"  Too-small (sigma_w^2=0.5):                        {np.round(var_low[::5], 8)}")
print(f"  Too-large (sigma_w^2=8.0):                        {np.round(var_high[::5], 2)}")
print(f"\nHe's value keeps variance in a stable, bounded range across 25 layers; the too-small")
print(f"value drives it to (numerically) zero, and the too-large value drives it to explode")
print(f"by orders of magnitude -- exactly the vanishing/exploding-signal failure modes this")
print(f"specific choice of sigma_w^2 is designed to avoid.")

# --- Quantify the order/chaos correlation map's phase transition ---
def propagate_correlation(sigma_w_sq, sigma_b_sq, n_layers, width, activation, c0, n_trials, rng):
    correlations = []
    for trial in range(n_trials):
        x1 = rng.normal(size=width)
        x1 = x1 / np.linalg.norm(x1) * np.sqrt(width)
        noise = rng.normal(size=width)
        noise = noise / np.linalg.norm(noise) * np.sqrt(width)
        x2 = c0 * x1 + np.sqrt(max(1 - c0**2, 0)) * noise
        layer_corrs = [np.corrcoef(x1, x2)[0, 1]]
        for layer in range(n_layers):
            W = rng.normal(scale=np.sqrt(sigma_w_sq / width), size=(width, width))
            b = rng.normal(scale=np.sqrt(sigma_b_sq), size=width)
            z1, z2 = W @ x1 + b, W @ x2 + b
            x1, x2 = activation(z1), activation(z2)
            layer_corrs.append(np.corrcoef(z1, z2)[0, 1])
        correlations.append(layer_corrs)
    return np.mean(correlations, axis=0)

print("\nCorrelation-map phase transition, tanh network, starting from c0=0.3:")
for sigma_w_sq in [0.8, 1.05, 1.3, 3.0]:
    corr_traj = propagate_correlation(sigma_w_sq, 0.05, n_layers=20, width=250, activation=np.tanh,
                                       c0=0.3, n_trials=25, rng=rng)
    depth_to_99 = next((i for i, c in enumerate(corr_traj) if c > 0.99), None)
    print(f"  sigma_w^2={sigma_w_sq}: final correlation (depth 20) = {corr_traj[-1]:.4f}, "
          f"reaches >0.99 by layer {depth_to_99 if depth_to_99 is not None else 'never'}")
print(f"\nSmaller sigma_w^2 collapses correlation to >0.99 within just a few layers (ordered")
print(f"phase); at sigma_w^2=3.0, correlation never approaches 1 even after 20 layers")
print(f"(chaotic phase) -- a genuine qualitative phase difference, not just a quantitative one.")

## 🤖 Why This Matters for AI

This is the theory behind principled weight initialization schemes used in virtually every deep
network today — **He initialization** ($\sigma_w^2=2/n$, for ReLU) and **Glorot/Xavier
initialization** ($\sigma_w^2 = 1/n$ or $2/(n_{in}+n_{out})$, originally derived for
$\tanh$/sigmoid) are both *exactly* the specific $\sigma_w^2$ values that keep signal
propagation stable — avoiding the vanishing/exploding variance this notebook demonstrated
directly above, and sitting at or near the edge-of-chaos correlation-map transition. This
isn't just a story about activations at initialization: the same forward-propagation
instabilities have a mirror-image backward-propagation counterpart in the gradients (recall the
vanishing-gradient problem from `06`), so getting initialization right is directly what makes a
given depth *trainable at all*, before a single gradient step is taken. Below, we close the loop
started in the Theory section: train real, deep $\tanh$ networks at several $\sigma_w^2$ values
on an actual classification task, and confirm directly that a network initialized deep in the
ordered phase fails to learn (gradient near zero, loss stuck), while initializations at or past
the edge of chaos train successfully to near-zero loss.

In [ ]:
torch.manual_seed(0)

def make_tanh_mlp(width, depth, sigma_w_sq, sigma_b_sq, seed):
    torch.manual_seed(seed)
    layers = []
    d_in = 10
    for _ in range(depth):
        lin = nn.Linear(d_in, width)
        with torch.no_grad():
            lin.weight.normal_(0, np.sqrt(sigma_w_sq / d_in))
            lin.bias.normal_(0, np.sqrt(sigma_b_sq))
        layers += [lin, nn.Tanh()]
        d_in = width
    out = nn.Linear(d_in, 1)
    with torch.no_grad():
        out.weight.normal_(0, np.sqrt(sigma_w_sq / d_in))
        out.bias.zero_()
    layers.append(out)
    return nn.Sequential(*layers)

rng_ai = np.random.default_rng(0)
X_ai = torch.tensor(rng_ai.normal(size=(200, 10)), dtype=torch.float32)
w_true_ai = rng_ai.normal(size=10)
y_ai = torch.tensor((X_ai.numpy() @ w_true_ai > 0).astype(np.float32)).reshape(-1, 1)

def train_and_track(sigma_w_sq, depth=15, width=64, seed=1, n_steps=150):
    model = make_tanh_mlp(width, depth, sigma_w_sq, 0.05, seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    losses, first_layer_grad_norms = [], []
    for step in range(n_steps):
        opt.zero_grad()
        out = model(X_ai)
        loss = loss_fn(out, y_ai)
        loss.backward()
        first_layer_grad_norms.append(model[0].weight.grad.norm().item())
        opt.step()
        losses.append(loss.item())
    return losses, first_layer_grad_norms

sigma_values = [0.5, 1.0, 1.3, 3.0]  # spans deep-ordered, near-critical, and chaotic phases
results = {}
for sigma_w_sq in sigma_values:
    losses, grad_norms = train_and_track(sigma_w_sq)
    results[sigma_w_sq] = (losses, grad_norms)
    print(f"sigma_w^2={sigma_w_sq}: initial loss={losses[0]:.4f}, final loss={losses[-1]:.4f}, "
          f"first-layer grad norm at step 0: {grad_norms[0]:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {0.5: "#C44E52", 1.0: "#DD8452", 1.3: "#55A868", 3.0: "#4C72B0"}
for sigma_w_sq in sigma_values:
    losses, grad_norms = results[sigma_w_sq]
    axes[0].plot(losses, color=colors[sigma_w_sq], lw=2, label=f"$\\sigma_w^2$={sigma_w_sq}")
    axes[1].semilogy(grad_norms, color=colors[sigma_w_sq], lw=2, label=f"$\\sigma_w^2$={sigma_w_sq}")
axes[0].set_xlabel("training step"); axes[0].set_ylabel("loss")
axes[0].set_title("Training loss")
axes[1].set_xlabel("training step"); axes[1].set_ylabel("first-layer gradient norm (log scale)")
axes[1].set_title("Gradient magnitude reaching the FIRST layer\nof a 15-layer network")
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"\nsigma_w^2=0.5 (deep in the ordered phase, per the correlation-map results above) has a")
print(f"first-layer gradient that stays tiny throughout training -- the network never learns")
print(f"(loss stuck near {results[0.5][0][-1]:.3f}, close to random-guessing's ln(2)~0.693).")
print(f"Every setting at or past the edge of chaos trains successfully to near-zero loss.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a ReLU layer $z=Wx+b$ with $W_{ij}\sim\mathcal{N}(0,\sigma_w^2/n)$ i.i.d. and
   $\sigma_b^2=0$, and pre-activation variance $q$ at some layer, work out
   $\mathbb{E}[\text{relu}(z)^2]$ by hand for a single (scalar) pre-activation
   $z\sim\mathcal{N}(0,q)$, then use it to write the one-step recursion $q^{(\ell+1)} =
   \sigma_w^2\,\mathbb{E}[\phi(z)^2]$ purely in terms of $q^{(\ell)}$ and $\sigma_w^2$. As a
   numerical sanity check, confirm your formula predicts $q$ stays exactly constant at
   $\sigma_w^2=2$ (He's value) and confirm this matches the notebook's own `var_he` values
   staying in a bounded range rather than trending toward 0 or $\infty$.

<details>
<summary>💡 Solution</summary>

For $Z\sim\mathcal{N}(0,q)$, split the integral at $0$ (ReLU zeroes out the negative half):
$$\mathbb{E}[\text{relu}(Z)^2] = \int_0^\infty z^2\frac{1}{\sqrt{2\pi q}}e^{-z^2/2q}\,dz =
\tfrac{1}{2}\mathbb{E}[Z^2] = \tfrac{q}{2},$$
using that $z^2$ is even and ReLU keeps exactly the half of the Gaussian's mass where $z>0$, so
the restricted integral is exactly half of the full second moment $\mathbb{E}[Z^2]=q$. Since
$\sigma_b^2=0$ here, the recursion becomes
$$q^{(\ell+1)} = \sigma_w^2\cdot\frac{q^{(\ell)}}{2} = \frac{\sigma_w^2}{2}\,q^{(\ell)}.$$
This is a simple geometric recursion: $q$ shrinks each layer if $\sigma_w^2<2$, grows if
$\sigma_w^2>2$, and — the He-initialization sweet spot — stays *exactly fixed* at
$\sigma_w^2=2$, since $q^{(\ell+1)}=\frac{2}{2}q^{(\ell)}=q^{(\ell)}$. Checking against the
notebook's own `var_he` (empirical, finite-width simulation): it stays bounded roughly in
$[1,2.5]$ across 25 layers rather than collapsing toward 0 (like `var_low`,
$\sigma_w^2=0.5$, which the recursion correctly predicts halves every layer) or exploding
(like `var_high`, $\sigma_w^2=8$, which quadruples every layer) — matching the qualitative
prediction, with the residual gap from the idealized recursion's exact "stays fixed" being
finite-width noise (this notebook's own mean-field approximation, not an error).

</details>

### 💭 UNDERSTAND
2. The Implementation section reports `var_low` (ReLU, $\sigma_w^2=0.5$) printing as exactly
   `0.0` after enough layers. Explain precisely why this is floating-point underflow rather than
   a genuine mathematical zero, and why ReLU specifically makes the collapse a one-way trap:
   once a unit's activation underflows to exactly $0.0$, can it ever recover a nonzero value in
   a later layer, and does the same argument apply to a `tanh` network's units collapsing toward
   (but never reaching) a fixed point?

<details>
<summary>💡 Solution</summary>

By the COMPUTE recursion, ReLU variance with $\sigma_w^2=0.5$ *exactly* halves every layer:
$q^{(\ell)} = q^{(0)}(1/4)^\ell$ (since $\sigma_w^2/2=0.25$ here) — mathematically this only
approaches $0$ in the limit, never truly reaching it. Float64 has roughly $10^{-308}$ as its
smallest positive normal magnitude; once $q^{(\ell)}$ drops below that (which the recursion says
happens by around layer 25 or so, since $0.25^{25}\approx 10^{-15}$ relative to a starting
variance of order 1, and further layers keep compounding down), every one of that layer's
activations rounds to the literal floating-point value `0.0`, and `np.var` of an all-zero array
is exactly `0.0`, not merely small. ReLU makes this permanent: `relu(W @ 0 + 0) = relu(0) = 0`
exactly, for *any* $W$ — once the input to a layer is the zero vector, every subsequent layer's
output is the zero vector too, since ReLU and matrix multiplication both map $\mathbf{0}\to
\mathbf{0}$. A `tanh` network's collapse is qualitatively different: variance/correlation
approach a fixed point exponentially but the underlying pre-activations remain (in exact
arithmetic) distinct nonzero real numbers that are merely extremely close together — `tanh`
never maps two different nonzero inputs to bit-identical outputs the way underflow forces ReLU's
outputs to become bit-identical zeros, so the "trap" here is specifically a floating-point
artifact of ReLU's exact-zero fixed point, not a universal feature of information collapse.

</details>

### ✏️ DERIVE
3. For the *linear* (identity-activation) case $\phi(z)=z$ — a useful simplified limit, since it
   removes the activation-specific expectation calculation — derive the exact closed-form
   solution $q^{(\ell)}$ as a function of layer $\ell$, $\sigma_w^2$, $\sigma_b^2$, and
   $q^{(0)}$, and identify the precise condition on $\sigma_w^2$ under which $q^{(\ell)}$
   converges to a finite, nonzero fixed point as $\ell\to\infty$.

<details>
<summary>💡 Solution outline</summary>

With $\phi(z)=z$, $\mathbb{E}[\phi(z)^2] = \mathbb{E}[z^2] = q^{(\ell)}$ exactly (no
activation-induced factor of $\tfrac12$ or similar), so the recursion is
$$q^{(\ell+1)} = \sigma_w^2\,q^{(\ell)} + \sigma_b^2.$$
This is a standard first-order linear recurrence. Its solution (by unrolling, or by solving for
the fixed point and adding the homogeneous solution) is
$$q^{(\ell)} = (\sigma_w^2)^\ell\Big(q^{(0)} - q^\star\Big) + q^\star, \qquad
q^\star = \frac{\sigma_b^2}{1-\sigma_w^2}\ \ (\sigma_w^2\ne 1),$$
where $q^\star$ is the fixed point solving $q^\star=\sigma_w^2 q^\star+\sigma_b^2$. Convergence
to this *finite* fixed point requires $|\sigma_w^2|<1$, so that $(\sigma_w^2)^\ell\to0$; at
$\sigma_w^2=1$ (with $\sigma_b^2>0$) $q^{(\ell)}$ instead grows linearly ($q^{(\ell)}=q^{(0)} +
\ell\sigma_b^2$), and for $\sigma_w^2>1$ it diverges geometrically. This linear case is a useful
point of comparison to the notebook's ReLU recursion (COMPUTE): ReLU's extra factor of
$\tfrac12$ from clipping negative values is exactly why ReLU's *critical* $\sigma_w^2$ is $2$
rather than $1$ — the activation function directly shifts where the stable/unstable boundary
sits, even though the *shape* of the recursion (linear, first-order) is the same in both cases.

</details>

### 🐍 IMPLEMENT
4. Add a residual/skip-connection variant of `propagate_variance`: instead of `x = activation(z)`
   at each layer, use `x = x + activation(z)` (so the layer's output is added to, not replacing,
   its input — the mean-field analogue of a ResNet block). Run it at the "too-small" setting
   ($\sigma_w^2=0.5$, ReLU) that collapses to zero in the plain (non-residual) version, and check
   what happens to the pre-activation variance across the same number of layers.

<details>
<summary>✅ How to know you're right</summary>

With the skip connection, variance should no longer collapse toward zero at $\sigma_w^2=0.5$ —
instead of shrinking by the plain network's exact factor of $\sigma_w^2/2=0.25$ per layer
(underflowing to bit-exact `0.0` well before layer 25, as in COMPUTE/UNDERSTAND), the residual
version's variance should now visibly *grow* across the same 25 layers (each layer adds a new,
independent contribution on top of whatever the running sum already was, so the sum can only
increase). The specific numbers will depend on your random seed, but the qualitative check that
matters is directional: plain-network final variance should be at or extremely close to
`0.0` (underflowed), while the residual version's final variance should be many orders of
magnitude larger than its initial variance — direct empirical confirmation of why skip
connections (central to ResNets, and to every transformer block via its residual stream) are
such an effective complementary fix for exactly the failure mode `11.04`'s Theory section
describes, independent of tuning $\sigma_w^2$ itself.

</details>

### 🤖 APPLY
5. Xavier/Glorot initialization ($\sigma_w^2=1$, historically derived for $\tanh$/sigmoid) is
   sometimes applied by mistake to ReLU networks (which call for He's $\sigma_w^2=2$). Using
   `propagate_variance`, compare Xavier's $\sigma_w^2=1$ against He's $\sigma_w^2=2$, both with
   the `relu` activation, across the same depth.

<details>
<summary>✅ What you should observe</summary>

Xavier's $\sigma_w^2=1$ on ReLU should show clear, one-directional variance decay — by the
COMPUTE recursion, $q^{(\ell+1)}=\frac{1}{2}q^{(\ell)}$ exactly, so variance halves every single
layer (empirically, dropping about 6 orders of magnitude by around layer 20, versus He's
$\sigma_w^2=2$ staying in a stable, bounded range, around 1-2.5, across the same depth). This is
a concrete illustration of why the "right" initialization variance is activation-dependent, not
a universal constant: Xavier's derivation implicitly assumes an activation (like $\tanh$ near
the origin) that behaves close to linear and roughly preserves variance in expectation, an
assumption ReLU violates by zeroing out exactly half of its input's mass — which is precisely
the extra factor of $\tfrac12$ that shifts ReLU's own critical value from $1$ up to $2$.

</details>

### 🚀 CHALLENGE
6. Push the AI section's phase-transition-vs-trainability experiment closer to the critical
   point: add $\sigma_w^2=1.05$ (near the edge-of-chaos value the correlation-map results
   suggest for this $\tanh$ setup) and increase `depth` from `15` to `30`. Run it across several
   random seeds (varying only the initialization seed, keeping $\sigma_w^2$ and depth fixed).
   Does this near-critical setting train reliably, train inconsistently, or fail outright,
   compared to a setting comfortably in the chaotic phase like $\sigma_w^2=1.3$? What does this
   suggest about why practitioners tend not to tune $\sigma_w^2$ to sit exactly at criticality?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports that near-critical initialization ($\sigma_w^2=1.05$) at this depth is
*unreliable* across seeds — in one investigation, final loss after 150 steps ranged from
essentially untrained (loss $\approx0.69$, matching random-guessing's $\ln 2$) to well-trained
(loss below $0.01$) purely as a function of the random initialization seed, with no other change
— whereas $\sigma_w^2=1.3$ (comfortably chaotic-side) trained successfully to near-zero loss
consistently across every seed tried, and deep-ordered settings ($\sigma_w^2=0.5$) failed to
train (loss stuck near $\ln 2$, first-layer gradients many orders of magnitude smaller) equally
consistently across every seed. This matches the theory: right at the phase boundary, whether a
*specific finite-width, finite-depth* network initialization behaves more like the ordered or
chaotic idealized limit is itself sensitive to the particular (finite) random draw, since both
"attracting" and "repelling" behavior near the fixed point are only exactly separated in the
strict infinite-width limit. A strong answer connects this to why real architectures don't
try to tune $\sigma_w^2$ to sit exactly at criticality: doing so would mean the network's
trainability becomes a matter of getting lucky with initialization rather than a robust,
seed-independent property, and this is exactly the practical gap that complementary techniques
mentioned in `06` and revisited in IMPLEMENT above — residual connections, and normalization
layers that actively re-center/re-scale activations every layer rather than relying on a single
static $\sigma_w^2$ choice to remain correct across all of training — are designed to close.

</details>

---

## 📚 Further Reading
- Poole et al., [\"Exponential expressivity in deep neural networks through transient chaos\"](https://arxiv.org/abs/1606.05340)
- Schoenholz et al., [\"Deep Information Propagation\"](https://arxiv.org/abs/1611.01232)
- He et al., [\"Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification\"](https://arxiv.org/abs/1502.01852)

---

*Next notebook: [Capstone: Build Attention from Math](../12_Capstone_Projects/01_build_attention_from_math.ipynb)*